In [ ]:
import os
import requests

CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Set SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET before running this cell.")

r = requests.post(
    "https://accounts.spotify.com/api/token",
    data={"grant_type": "client_credentials"},
    auth=(CLIENT_ID, CLIENT_SECRET),
)

print("status:", r.status_code)
r.raise_for_status()

token = r.json()["access_token"]
print("Spotify access token retrieved.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/regional-global-weekly-2026-02-19.csv"  # <-- change if needed
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
print("rows:", len(df))
print("columns:", df.columns.tolist())
df.head()

rows: 200
columns: ['rank', 'uri', 'artist_names', 'track_name', 'source', 'peak_rank', 'previous_rank', 'weeks_on_chart', 'streams']


,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
0,1,spotify:track:3sK8wGT43QFpWrvNQsrQya,Bad Bunny,DtMF,Rimas Entertainment LLC.,1,1,59,60572294
1,2,spotify:track:2lTm559tuIvatlT1u0JYG2,Bad Bunny,BAILE INoLVIDABLE,Rimas Entertainment LLC.,2,2,59,43887047
2,3,spotify:track:5TFD2bmFKGhoCRbX61nXY5,Bad Bunny,NUEVAYoL,Rimas Entertainment LLC.,3,3,59,43135702
3,4,spotify:track:1IHWl5LamUGEuP4ozKQSXZ,Bad Bunny,Tití Me Preguntó,Rimas Entertainment LLC,4,8,64,39762842
4,5,spotify:track:3qhlB30KknSejmIvZZLjOD,Djo,End of Beginning,Djo,1,5,105,35225040


In [ ]:
df["track_id"] = df["uri"].astype(str).str.replace("spotify:track:", "", regex=False)
df2 = df[df["track_id"].str.len().gt(10)].copy()

print("Rows with track_id:", len(df2))
df2[["uri","track_id"]].head()

Rows with track_id: 200


,uri,track_id
0,spotify:track:3sK8wGT43QFpWrvNQsrQya,3sK8wGT43QFpWrvNQsrQya
1,spotify:track:2lTm559tuIvatlT1u0JYG2,2lTm559tuIvatlT1u0JYG2
2,spotify:track:5TFD2bmFKGhoCRbX61nXY5,5TFD2bmFKGhoCRbX61nXY5
3,spotify:track:1IHWl5LamUGEuP4ozKQSXZ,1IHWl5LamUGEuP4ozKQSXZ
4,spotify:track:3qhlB30KknSejmIvZZLjOD,3qhlB30KknSejmIvZZLjOD


In [ ]:
import pandas as pd, requests, time, re

headers = {"Authorization": f"Bearer {token}"}

def get_json(url, params=None, max_retries=8):
    for _ in range(max_retries):
        r = requests.get(url, headers=headers, params=params)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "2"))
            time.sleep(wait + 0.2)
            continue
        if r.status_code >= 400:
            return None
        return r.json()
    return None

def primary_artist_name(s):
    s = str(s)
    s = re.split(r",| & | feat\. | ft\. | featuring ", s, flags=re.IGNORECASE)[0]
    return s.strip()

df = df.copy()
df["primary_artist"] = df["artist_names"].apply(primary_artist_name)

unique_artists = df["primary_artist"].dropna().unique().tolist()
print("Unique artists:", len(unique_artists))

artist_to_genre = {}
sleep_s = 0.12   # ✅ faster; if you still get many 429s, increase to 0.2

for a in unique_artists:
    if a in artist_to_genre:
        continue

    sdata = get_json("https://api.spotify.com/v1/search",
                     params={"q": f'artist:"{a}"', "type": "artist", "limit": 5, "market": "US"})
    artist_id = None
    if sdata and "artists" in sdata and sdata["artists"].get("items"):
        artist_id = sdata["artists"]["items"][0].get("id")

    genre = "unknown"
    if artist_id:
        adata = get_json(f"https://api.spotify.com/v1/artists/{artist_id}")
        g = (adata.get("genres") or []) if adata else []
        genre = g[0] if g else "unknown"

    artist_to_genre[a] = genre
    time.sleep(sleep_s)

df["genre"] = df["primary_artist"].map(artist_to_genre)
df[["rank","artist_names","track_name","streams","genre"]].head(10)

Unique artists: 129


KeyboardInterrupt: 

In [ ]:
import requests, time

token = token  # your existing token string
headers = {"Authorization": f"Bearer {token}"}

TRACK_ID = "3sK8wGT43QFpWrvNQsrQya"  # DtMF

def get_once(url, params=None, timeout=20):
    r = requests.get(url, headers=headers, params=params, timeout=timeout)
    if r.status_code == 429:
        ra = int(r.headers.get("Retry-After", "0"))
        raise RuntimeError(f"Rate-limited (429). Retry-After={ra}s (≈{ra/3600:.1f} hours). Stop and change IP/app/token.")
    r.raise_for_status()
    return r.json()

# 1) track -> primary artist
t = get_once(f"https://api.spotify.com/v1/tracks/{TRACK_ID}", params={"market":"US"})
artist = t["artists"][0]
artist_id = artist["id"]
artist_name = artist["name"]
print("Artist:", artist_name, artist_id)

# 2) artist -> genres (Spotify “genre”)
a = get_once(f"https://api.spotify.com/v1/artists/{artist_id}")
genres = a.get("genres") or []
print("Spotify genres:", genres)
print("Primary genre:", genres[0] if genres else "unknown (Spotify returned empty/missing genres)")

RuntimeError: Rate-limited (429). Retry-After=84132s (≈23.4 hours). Stop and change IP/app/token.